# Sensor data exploration

One participant. Raw sensor/questionnaire data cover Session 1; supplied eye summaries cover three sessions. These plots are descriptive, with no anxiety classifier or accuracy estimate. Channel-to-device mapping, clock synchronization, pupil calibration and summary derivation are undocumented. Interval readings are not beat-to-beat cardiac data.

`CSV → legacy low-movement runs → descriptive plots`

See the [data dictionary](../data/DATA_DICTIONARY.md) for source columns and limitations.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import sensor_data as sd

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["figure.dpi"] = 100

COLORS = {
    "HADS": "#e74c3c",
    "STAI-S": "#e67e22",
    "STAI-T": "#2ecc71",
    "BFI": "#9b59b6",
    "FQ": "#3498db",
}

## 1. Load everything

In [ ]:
data = sd.load_all()
hr, ibi, sed_fix, eye_metrics, psych = (
    data[k] for k in ["hr", "ibi", "sed_fix", "eye_metrics", "psych"]
)

print(f"hr:      {hr.shape[0]:>6,} rows")
print(f"ibi:     {ibi.shape[0]:>6,} rows")
print(f"sed_fix: {sed_fix.shape[0]:>6,} rows")
print(f"eye_metrics:     {eye_metrics.shape[0]:>6,} rows  (3 sessions)")
print(f"psych:   {psych.shape[0]:>6,} rows  (session 1 only)")

## 2. Heart rate over recording time

Anonymous channels 3 and 5 contain readings. The plot retains `confidence == 1.0`; the source does not establish which device produced each channel.

In [ ]:
hr_valid = sd.valid_hr(hr)

fig, ax = plt.subplots()
for sensor_id, group in hr_valid.groupby("iSensor"):
    ax.plot(
        group["reltime"],
        group["heart_rate"],
        label=f"Channel {sensor_id}",
        alpha=0.7,
        linewidth=0.8,
    )

ax.set_xlabel("Recording time (seconds)")
ax.set_ylabel("Reported heart rate (BPM)")
ax.set_title("Heart-rate readings: Session 1")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Reported pupil measurements

Source pupil values, with a trailing two-second mean requiring ten valid observations. Nonfinite/nonpositive pupil values or pupil quality outside `(0.5, 1]` are masked; invalid rows and gaps over two seconds break the curves. Millimetres are reported units, with calibration unverified. Questionnaire boundaries are not overlaid because shared clock/time-zone provenance is missing.

In [ ]:
def break_gaps(series):
    """Insert display breaks across gaps longer than two seconds."""
    times = series.index.asi8
    gaps = np.flatnonzero(np.diff(times) > pd.Timedelta(seconds=2).value)
    middle = times[gaps] + (times[gaps + 1] - times[gaps]) // 2
    breaks = pd.Series(np.nan, index=pd.to_timedelta(middle, unit="ns"))
    return pd.concat([series, breaks]).sort_index(kind="stable")


def pupil_trace(eye, min_periods=10):
    """Require real measurements and smooth valid rows over two seconds."""
    for column in ("reltime", "pupil", "pupilQ"):
        if pd.api.types.is_complex_dtype(eye[column]) or any(
            isinstance(value, (complex, np.complexfloating)) for value in eye[column]
        ):
            raise ValueError(f"{column} must contain real measurements")
    relative = eye["reltime"]
    if not pd.api.types.is_numeric_dtype(relative) or pd.api.types.is_bool_dtype(relative):
        raise ValueError("reltime must be numeric")
    if (
        not np.isfinite(relative).all()
        or (relative < 0).any()
        or not relative.is_monotonic_increasing
    ):
        raise ValueError("reltime must be finite, nonnegative and nondecreasing")
    times = pd.to_timedelta(relative, unit="s")
    valid = (
        np.isfinite(eye["pupil"])
        & np.isfinite(eye["pupilQ"])
        & (eye["pupil"] > 0)
        & (eye["pupilQ"] > 0.5)
        & (eye["pupilQ"] <= 1)
    )
    raw = pd.Series(eye["pupil"].to_numpy(), index=times).where(valid.to_numpy())
    mean = raw.rolling("2s", min_periods=min_periods).mean().where(valid.to_numpy())
    return break_gaps(raw), break_gaps(mean)


pupil_raw, pupil_mean = pupil_trace(sed_fix)
fig, ax = plt.subplots()
ax.plot(
    pupil_raw.index.total_seconds(), pupil_raw.values, color="steelblue", alpha=0.4, linewidth=0.3
)
ax.plot(
    pupil_mean.index.total_seconds(),
    pupil_mean.values,
    color="darkblue",
    linewidth=1.5,
    label="Trailing 2 s mean (minimum 10 readings)",
)
ax.set_xlabel("Recording time (seconds)")
ax.set_ylabel("Reported pupil value (mm)")
ax.set_title("Pupil readings: Session 1")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 4. Legacy low-movement run durations

`sed_fix.csv` labels consecutive gaze-vector displacement below 0.01 as `fixation`. This unvalidated rule has no quality, gap or minimum-duration filter. The histogram uses one elapsed span per positive-duration run; it does not establish physiological fixations or an anxiety threshold.

In [ ]:
run_spans = sd.fixation_durations(sed_fix)

fig, ax = plt.subplots()
ax.hist(run_spans * 1000, bins=80, color="steelblue", edgecolor="white", alpha=0.8)
ax.set_xlabel("Legacy low-movement run span (ms)")
ax.set_ylabel("Run count")
ax.set_title("Legacy run durations: Session 1")
plt.tight_layout()
plt.show()

## 5. Interval readings and arithmetic illustration

Channel 5 has the most interval readings. Values repeat and are quantized; these are not a measured sequence of normal-to-normal beats. The lower panel uses the same 30-reading window for the root mean square of 29 adjacent differences and sample standard deviation. It illustrates arithmetic on the supplied samples, not cardiac HRV or anxiety.

In [ ]:
def interval_variation(values, window=30):
    """Use one reading window for adjacent differences and sample SD."""
    rolling = values.rolling(window, min_periods=window)
    difference_rms = rolling.apply(lambda x: np.sqrt(np.mean(np.diff(x) ** 2)), raw=True)
    return difference_rms, rolling.std(ddof=1)


ibi_s5 = ibi[(ibi["iSensor"] == 5) & (ibi["ibi"] > 0)].copy().reset_index(drop=True)
difference_rms, sample_sd = interval_variation(ibi_s5["ibi"])
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(ibi_s5["reltime"], ibi_s5["ibi"], color="steelblue", linewidth=0.6)
axes[0].set_ylabel("Reported interval (ms)")
axes[0].set_title("Interval readings: Channel 5")
axes[1].plot(
    ibi_s5["reltime"], difference_rms, label="Adjacent-difference RMS", color="#e74c3c", linewidth=1
)
axes[1].plot(ibi_s5["reltime"], sample_sd, label="Sample SD", color="#3498db", linewidth=1)
axes[1].set_xlabel("Recording time (seconds)")
axes[1].set_ylabel("Illustrative variation (ms)")
axes[1].set_title("Arithmetic illustration: 30 readings, 29 differences; not measured HRV")
axes[1].legend()
plt.tight_layout()
plt.show()

## 6. Supplied pupil summaries across sessions

Original `Average Pupil Dilation` values, grouped by test/session. Units are reported as mm; aggregation, baseline, calibration and blink definitions are unavailable. Values and flags are preserved, with no inferred cause for session differences.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for i, (test_name, ax) in enumerate(zip(["Test 01", "Test 02", "Test 03"], axes, strict=True)):
    session = eye_metrics[eye_metrics["Test"] == test_name]
    test_types = session["Type"].unique()

    x_offset = 0
    ticks, labels = [], []
    for t in test_types:
        subset = session[session["Type"] == t]
        positions = range(x_offset, x_offset + len(subset))
        ax.bar(
            positions,
            subset["Average Pupil Dilation"].values,
            color=COLORS.get(t, "gray"),
            alpha=0.8,
            width=0.9,
        )
        ticks.append(x_offset + len(subset) // 2)
        labels.append(t)
        x_offset += len(subset) + 1

    ax.set_xticks(ticks)
    ax.set_xticklabels(labels, rotation=45, fontsize=8)
    ax.set_title(test_name)
    if i == 0:
        ax.set_ylabel("Supplied pupil summary (reported mm)")

fig.suptitle("Supplied pupil summaries across sessions", fontsize=12)
plt.tight_layout()
plt.show()

## 7. Response times by test

Recorded response-time distributions for Session 1. They do not establish deliberation, avoidance or anxiety; item content, order and reading time are uncontrolled.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

test_order = ["HADS", "STAI-S", "STAI-T", "BFI", "FQ"]
data_by_test = [psych[psych["Test"] == t]["Time(s)"].values for t in test_order]

bp = ax.boxplot(data_by_test, tick_labels=test_order, patch_artist=True)
for patch, t in zip(bp["boxes"], test_order, strict=True):
    patch.set_facecolor(COLORS.get(t, "gray"))
    patch.set_alpha(0.6)

ax.set_ylabel("Response time (seconds)")
ax.set_title("Recorded response times: Session 1")
plt.tight_layout()
plt.show()

## Related work

[Multimodal-Multisensor](https://github.com/urmeo/Multimodal-Multisensor): longitudinal eye/HRV thesis. [CalmSense](https://github.com/urmeo/CalmSense): a separate wearable stress benchmark. Neither supplies validation of this sample's legacy rule.